In [1]:
!wget http://nlp.stanford.edu/data/glove.6B.zip

!unzip -q glove.6B.zip -d glove

--2026-03-04 06:09:35--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-03-04 06:09:35--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-03-04 06:09:35--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

# **Task 1 - Data Preparation**

In [4]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split

from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

nltk.download('punkt')
nltk.download('punkt_tab')
df = pd.read_csv('movies.csv')

df = df[['genres', 'keywords', 'tagline', 'overview', 'vote_average']]

df.rename(columns={
    'genres': 'genre',
    'vote_average': 'voting_average'
}, inplace=True)

df = df.dropna().reset_index(drop=True)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]

    return tokens

TEXT_COLUMNS = ['overview', 'tagline', 'keywords']

for col in TEXT_COLUMNS:
    df[col + '_clean'] = df[col].apply(clean_text)

df['genre_list'] = df['genre'].apply(lambda x: x.split())

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Train size: 2631
Validation size: 564
Test size: 564


# **Task 2 - GloVe Embedding Pipeline**

Embedding used: 100d

In [5]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

EMBEDDING_DIM = 100
GLOVE_PATH = "glove/glove.6B.100d.txt"
TEXT_COLUMNS = ['overview', 'tagline', 'keywords']

print("Loading GloVe 100D embeddings...")

glove = {}
with open(GLOVE_PATH, 'r', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        glove[word] = vector

print("Total GloVe words loaded:", len(glove))

unique_tokens = set()

for col in TEXT_COLUMNS:
    for tokens in df[col + '_clean']:
        unique_tokens.update(tokens)

covered_tokens = [word for word in unique_tokens if word in glove]
coverage = 100 * len(covered_tokens) / len(unique_tokens)

print("Unique dataset tokens:", len(unique_tokens))
print("Tokens found in GloVe:", len(covered_tokens))
print("Embedding Coverage: {:.2f}%".format(coverage))

def build_embeddings(text_column):

    train_texts = train_df[text_column + '_clean'].apply(lambda x: " ".join(x))
    val_texts   = val_df[text_column + '_clean'].apply(lambda x: " ".join(x))
    test_texts  = test_df[text_column + '_clean'].apply(lambda x: " ".join(x))

    vectorizer = TfidfVectorizer()
    vectorizer.fit(train_texts)

    feature_names = vectorizer.get_feature_names_out()

    def compute_doc_embedding(text_series):
        tfidf_matrix = vectorizer.transform(text_series)
        doc_embeddings = np.zeros((tfidf_matrix.shape[0], EMBEDDING_DIM))

        for i in range(tfidf_matrix.shape[0]):
            row = tfidf_matrix[i]
            indices = row.indices
            weights = row.data

            weighted_sum = np.zeros(EMBEDDING_DIM)
            weight_total = 0

            for idx, weight in zip(indices, weights):
                word = feature_names[idx]
                if word in glove:
                    weighted_sum += weight * glove[word]
                    weight_total += weight

            if weight_total > 0:
                doc_embeddings[i] = weighted_sum / weight_total

        return doc_embeddings

    X_train_col = compute_doc_embedding(train_texts)
    X_val_col   = compute_doc_embedding(val_texts)
    X_test_col  = compute_doc_embedding(test_texts)

    return X_train_col, X_val_col, X_test_col

X_train = {}
X_val = {}
X_test = {}

for col in TEXT_COLUMNS:
    print(f"\nBuilding embeddings for: {col}")
    X_train[col], X_val[col], X_test[col] = build_embeddings(col)

    print("Train shape:", X_train[col].shape)
    print("Validation shape:", X_val[col].shape)
    print("Test shape:", X_test[col].shape)

print("\nTask 2 Completed Successfully.")

Loading GloVe 100D embeddings...
Total GloVe words loaded: 400000
Unique dataset tokens: 21405
Tokens found in GloVe: 19137
Embedding Coverage: 89.40%

Building embeddings for: overview
Train shape: (2631, 100)
Validation shape: (564, 100)
Test shape: (564, 100)

Building embeddings for: tagline
Train shape: (2631, 100)
Validation shape: (564, 100)
Test shape: (564, 100)

Building embeddings for: keywords
Train shape: (2631, 100)
Validation shape: (564, 100)
Test shape: (564, 100)

Task 2 Completed Successfully.


# **Task 3 - Model A: Rating Prediction (Regression)**

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error
import numpy as np

y_train_reg = train_df['voting_average'].values
y_test_reg  = test_df['voting_average'].values

def get_doc_embeddings_for_col(text_col):
    """
    Returns (X_train_col, X_test_col) as numpy arrays.
    text_col may be 'overview', 'overview_clean', 'tagline', 'tagline_clean', etc.
    """
    key = text_col
    if key.endswith('_clean'):
        key = key[:-6]

    if key not in X_train:
        raise KeyError(f"Column '{key}' not found in X_train. Available keys: {list(X_train.keys())}")

    X_train_col = X_train[key]
    X_test_col  = X_test[key]
    return X_train_col, X_test_col

class RegressionNet(nn.Module):
    def __init__(self):
        super(RegressionNet, self).__init__()

        self.fc1 = nn.Linear(100, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

def train_and_evaluate_regression(text_col):
    print(f"--- Training Model A (Regression) using: {text_col} ---")

    X_train_col, X_test_col = get_doc_embeddings_for_col(text_col)

    train_dataset = TensorDataset(torch.tensor(X_train_col, dtype=torch.float32),
                                  torch.tensor(y_train_reg, dtype=torch.float32).unsqueeze(1))
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    model = RegressionNet()
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

    epochs = 25
    for epoch in range(epochs):
        epoch_loss = 0.0
        model.train()
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            predictions = model(inputs)
            loss = criterion(predictions, targets)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            avg_loss = epoch_loss / len(train_loader)
            print(f"Epoch {epoch+1}/{epochs} | Training Loss: {avg_loss:.4f}")


    model.eval()
    with torch.no_grad():
        test_preds = model(torch.tensor(X_test_col, dtype=torch.float32)).numpy()


    mse = mean_squared_error(y_test_reg, test_preds)
    rmse = np.sqrt(mse)

    global_mean = train_df['voting_average'].mean()
    baseline_preds = np.full_like(y_test_reg, global_mean)
    baseline_mse = mean_squared_error(y_test_reg, baseline_preds)

    print(f"-> Baseline MSE: {baseline_mse:.4f}")
    print(f"-> Model MSE:    {mse:.4f}")
    print(f"-> Model RMSE:   {rmse:.4f}\n")

train_and_evaluate_regression('overview_clean')
train_and_evaluate_regression('tagline_clean')

--- Training Model A (Regression) using: overview_clean ---
Epoch 5/25 | Training Loss: 1.0668
Epoch 10/25 | Training Loss: 0.9165
Epoch 15/25 | Training Loss: 0.8577
Epoch 20/25 | Training Loss: 0.8215
Epoch 25/25 | Training Loss: 0.8233
-> Baseline MSE: 0.8977
-> Model MSE:    0.8508
-> Model RMSE:   0.9224

--- Training Model A (Regression) using: tagline_clean ---
Epoch 5/25 | Training Loss: 1.0430
Epoch 10/25 | Training Loss: 0.8791
Epoch 15/25 | Training Loss: 0.7943
Epoch 20/25 | Training Loss: 0.7248
Epoch 25/25 | Training Loss: 0.6338
-> Baseline MSE: 0.8977
-> Model MSE:    1.1758
-> Model RMSE:   1.0843



# **Task 4 - Model B: Genre Prediction (Multi-Label Classification)**

In [7]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, hamming_loss, jaccard_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

mlb = MultiLabelBinarizer()
y_train_cls = mlb.fit_transform(train_df['genre_list'])
y_val_cls   = mlb.transform(val_df['genre_list'])
y_test_cls  = mlb.transform(test_df['genre_list'])

num_classes = len(mlb.classes_)
print(f"Number of genres (classes): {num_classes}")
print("Genres:", mlb.classes_)

def _col_key_from_arg(text_col_arg):
    """
    Accept either 'overview' or 'overview_clean' etc and return the key used in X_train dict.
    """
    key = text_col_arg
    if key.endswith('_clean'):
        key = key[:-6]
    return key

def get_embeddings_for_col(text_col_arg):
    key = _col_key_from_arg(text_col_arg)
    if key not in X_train:
        raise KeyError(f"Column '{key}' not found in X_train. Available: {list(X_train.keys())}")
    return X_train[key], X_val[key], X_test[key]

class MultiLabelNet(nn.Module):
    def __init__(self, num_classes):
        super(MultiLabelNet, self).__init__()
        self.fc1 = nn.Linear(100, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

def train_and_evaluate_classification(text_col_arg):
    print(f"\n--- Training Model B (Classification) using: {text_col_arg} ---")
    Xtr, Xv, Xt = get_embeddings_for_col(text_col_arg)

    if not np.any(np.abs(Xtr)) :
        Xtr = Xtr + 1e-6
    train_dataset = TensorDataset(torch.tensor(Xtr, dtype=torch.float32),
                                  torch.tensor(y_train_cls, dtype=torch.float32))
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    model = MultiLabelNet(num_classes).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

    epochs = 20
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for inputs, targets in train_loader:
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            avg_loss = epoch_loss / len(train_loader)
            print(f"Epoch {epoch+1}/{epochs} | Training Loss: {avg_loss:.4f}")

    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(Xt, dtype=torch.float32).to(DEVICE))
        probs = torch.sigmoid(logits).cpu().numpy()

    preds = (probs > 0.5).astype(int)

    micro_f1 = f1_score(y_test_cls, preds, average='micro', zero_division=0)
    macro_f1 = f1_score(y_test_cls, preds, average='macro', zero_division=0)
    h_loss = hamming_loss(y_test_cls, preds)
    try:
        jacc = jaccard_score(y_test_cls, preds, average='samples', zero_division=0)
    except Exception:
        jacc = None

    print(f"-> Micro-F1:     {micro_f1:.4f}")
    print(f"-> Macro-F1:     {macro_f1:.4f}")
    print(f"-> Hamming Loss: {h_loss:.4f}")
    if jacc is not None:
        print(f"-> Jaccard Score: {jacc:.4f}")

    return {'micro_f1': micro_f1, 'macro_f1': macro_f1, 'hamming_loss': h_loss, 'jaccard': jacc, 'preds': preds, 'probs': probs, 'model': model}

results_overview = train_and_evaluate_classification('overview_clean')
results_tagline  = train_and_evaluate_classification('tagline_clean')


Device: cpu
Number of genres (classes): 22
Genres: ['Action' 'Adventure' 'Animation' 'Comedy' 'Crime' 'Documentary' 'Drama'
 'Family' 'Fantasy' 'Fiction' 'Foreign' 'History' 'Horror' 'Movie' 'Music'
 'Mystery' 'Romance' 'Science' 'TV' 'Thriller' 'War' 'Western']

--- Training Model B (Classification) using: overview_clean ---
Epoch 5/20 | Training Loss: 0.2197
Epoch 10/20 | Training Loss: 0.2004
Epoch 15/20 | Training Loss: 0.1866
Epoch 20/20 | Training Loss: 0.1736
-> Micro-F1:     0.5515
-> Macro-F1:     0.3883
-> Hamming Loss: 0.1008
-> Jaccard Score: 0.4207

--- Training Model B (Classification) using: tagline_clean ---
Epoch 5/20 | Training Loss: 0.2655
Epoch 10/20 | Training Loss: 0.2353
Epoch 15/20 | Training Loss: 0.2089
Epoch 20/20 | Training Loss: 0.1866
-> Micro-F1:     0.3072
-> Macro-F1:     0.1510
-> Hamming Loss: 0.1334
-> Jaccard Score: 0.2246


# **Task 5 - Frequent Words per Genre**

In [8]:
from collections import defaultdict, Counter
import pandas as pd

TEXT_COL = "overview_clean"

genre_word_counts = defaultdict(Counter)

for _, row in train_df.iterrows():

    words = row[TEXT_COL]
    genres = row["genre_list"]

    for g in genres:
        genre_word_counts[g].update(words)

MIN_FREQ = 3
results = {}
for genre, counter in genre_word_counts.items():

    filtered = {w:c for w,c in counter.items() if c >= MIN_FREQ}
    sorted_words = sorted(filtered.items(), key=lambda x: x[1], reverse=True)
    top10 = sorted_words[:10]
    bottom10 = sorted_words[-10:]

    results[genre] = {
        "top": top10,
        "bottom": bottom10
    }

for genre in sorted(results.keys()):

    print("\n==============================")
    print("Genre:", genre)
    print("==============================")

    top_df = pd.DataFrame(results[genre]["top"], columns=["Word","Frequency"])
    bottom_df = pd.DataFrame(results[genre]["bottom"], columns=["Word","Frequency"])

    print("\nTop 10 Frequent Words")
    print(top_df.to_string(index=False))

    print("\nBottom 10 Least Frequent Words")
    print(bottom_df.to_string(index=False))


Genre: Action

Top 10 Frequent Words
 Word  Frequency
  new        104
world        101
 must         98
  one         97
  two         81
 find         75
  man         73
 life         73
agent         67
young         65

Bottom 10 Least Frequent Words
      Word  Frequency
     wiley          3
      jill          3
   seattle          3
      thor          3
      wade          3
immigrants          3
   averill          3
 cavendich          3
       tao          3
     bazil          3

Genre: Adventure

Top 10 Frequent Words
 Word  Frequency
world         95
 find         86
  new         83
 must         81
  one         66
young         65
  two         64
 life         59
 help         47
 save         47

Bottom 10 Least Frequent Words
     Word  Frequency
     drug          3
   taylor          3
     thor          3
     wade          3
assistant          3
   austin          3
cavendich          3
 redferne          3
   undead          3
    alice          3

Genre: An

# **Task 6 - Genre-Indicative Words Using TF-IDF**

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

TEXT_COL = "overview_clean"

train_texts = train_df[TEXT_COL].apply(lambda x: " ".join(x))

vectorizer = TfidfVectorizer(min_df=3)
X_train_tfidf = vectorizer.fit_transform(train_texts)

feature_names = np.array(vectorizer.get_feature_names_out())
genre_indicative_words = {}

for i, genre in enumerate(mlb.classes_):

    print("\n==============================")
    print("Genre:", genre)
    print("==============================")

    y = y_train_cls[:, i]

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train_tfidf, y)

    weights = clf.coef_[0]

    top_indices = np.argsort(weights)[-10:][::-1]

    top_words = feature_names[top_indices]
    top_weights = weights[top_indices]

    df = pd.DataFrame({
        "Word": top_words,
        "Weight": top_weights
    })

    print(df.to_string(index=False))

    genre_indicative_words[genre] = df


Genre: Action
      Word   Weight
     agent 2.258307
    forces 1.869343
     james 1.856844
      bond 1.843321
       cop 1.674017
    battle 1.533685
       cia 1.394342
      must 1.392677
     earth 1.378615
government 1.378309

Genre: Adventure
     Word   Weight
     bond 2.459092
adventure 1.730645
     find 1.718982
    earth 1.695728
  captain 1.652307
     save 1.504863
    world 1.484206
     must 1.451408
     park 1.419725
    quest 1.388796

Genre: Animation
     Word   Weight
    world 1.345070
adventure 1.337687
    human 1.246849
    shrek 1.228382
     land 1.223825
   dragon 1.194935
    named 1.066987
 animated 1.063333
   prince 1.045972
  journey 1.043358

Genre: Comedy
     Word   Weight
   comedy 2.343957
     best 1.576245
      big 1.550626
  wedding 1.434612
christmas 1.422331
     find 1.281703
  romance 1.278021
      guy 1.243387
   doesnt 1.221890
  friends 1.207909

Genre: Crime
     Word   Weight
   police 2.928882
     drug 2.549991
detective 2.3343